# 병원 호흡기 입원 예측 — 시계열 EDA와 모델 (CDC NHSN)

- 데이터: 미국 병원 주간 호흡기 입원 (20,837행 · **322열**)
- 목표: 독감 신규 입원 수 예측 — 다중 시계열(권역별)
- 흐름: 불러오기 → 학습 전 확인 → 시계열 EDA → 형식 변환 → 학습 → 해석
- 참고: 데이터 소개 CDC_NHSN_hospital.txt

- 이 데이터의 핵심
  · **컬럼 322개** → 필요한 것만 골라내기
  · **천 단위 쉼표** → 숫자가 문자열로 읽힘
  · **보고율 변동** → 값의 변화가 실제인지 보고 문제인지

## 1. 불러오기 — 천 단위 쉼표 주의

- 숫자에 쉼표가 있어("1,110") thousands 지정 없으면 문자열로 읽힘
- 그대로 두면 학습 자체가 안 됨

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("hrd.csv", low_memory=False,
                 thousands=",")          # 필수
print(df.shape)                          # (20837, 322)

# 문자열 컬럼이 3개면 정상 (날짜·지리·시즌)
print("문자열 컬럼 수:", df.select_dtypes(include="object").shape[1])

## 2. 학습 전 확인

### 2-1. 322개 컬럼에서 필요한 것만 (핵심)

- 컬럼명이 「지표유형 + 질환 + 대상」 규칙으로 만들어짐
  예: "Percent Hospitals Reporting Influenza Admissions"
- 실습에는 4개면 충분
  · Total Influenza Admissions → 타깃
  · Percent Hospitals Reporting ... → 보고율(품질 확인용)
  · Week Ending Date, Geographic aggregation → 시계열 3요소

In [ ]:
cols = ["Week Ending Date",
        "Geographic aggregation",
        "Total Influenza Admissions",
        "Percent Hospitals Reporting Influenza Admissions"]
small = df[cols].copy()
small["Week Ending Date"] = pd.to_datetime(small["Week Ending Date"])
print(small.shape)
small.head()

### 2-2. 지리 층위를 섞지 말 것

- 이 데이터에는 주(56) · HHS권역(10) · 전국(USA)이 함께 있음
- 전국은 주의 합계에 가까움 → 함께 넣으면 중복·스케일 상충
- 한 층위만 선택 (여기서는 권역 10개)

In [ ]:
g = small["Geographic aggregation"]
print("권역 행수:", g.str.startswith("Region").sum())
print("전국 행수:", (g=="USA").sum())

region = small[g.str.startswith("Region")].copy()
print("\n선택 후:", region.shape, "| 권역:",
      region["Geographic aggregation"].nunique())

### 2-3. 보고율 변동 = 구조적 단절 (중요)

- 병원의 보고 의무가 시기마다 바뀜
- 보고율이 낮은 기간의 입원 수 감소는
  **실제 감소인지 보고 감소인지 구분되지 않음**
- 먼저 보고율 추이를 확인

In [ ]:
usa = small[small["Geographic aggregation"]=="USA"].set_index("Week Ending Date")

usa["Percent Hospitals Reporting Influenza Admissions"].plot(
    figsize=(12,3), title="reporting rate (%) over time")
plt.axhline(80, color="red", linestyle="--", label="80%")
plt.legend()
plt.show()
# 중간에 크게 떨어지는 구간이 있는가?
# → 그 기간의 입원 수는 해석에 주의

## 3. 시계열 EDA

### 3-1. 계절성 — 겨울 유행

In [ ]:
(region.assign(m=region["Week Ending Date"].dt.month)
   .groupby("m")["Total Influenza Admissions"].mean()
   .plot(kind="bar", figsize=(8,3),
         title="mean influenza admissions by month"))
plt.show()
# 겨울(12~2월) 피크, 여름 저점

### 3-2. 권역별 추이

- 권역마다 유행 시점·규모가 비슷한가

In [ ]:
pivot = region.pivot_table(
    index="Week Ending Date",
    columns="Geographic aggregation",
    values="Total Influenza Admissions")

pivot.plot(figsize=(13,4), title="influenza admissions by HHS region")
plt.legend(fontsize=7, ncol=2)
plt.show()
# 권역별 규모 차이 + 유행 시점의 유사성 확인

## 4. 시계열 형식 변환

- 이미 주간 데이터 → 집계 불필요
- 주 종료일이 토요일 → freq="W-SAT" 

In [ ]:
from autogluon.timeseries import TimeSeriesDataFrame

ts_df = region.rename(columns={
    "Week Ending Date": "timestamp",
    "Geographic aggregation": "item_id",
    "Total Influenza Admissions": "target",
})[["item_id","timestamp","target"]].dropna(subset=["target"])

ts = TimeSeriesDataFrame.from_data_frame(
    ts_df, id_column="item_id", timestamp_column="timestamp")
ts = ts.convert_frequency(freq="W-SAT")
print("변환 완료:", ts.shape)

## 5. 학습

- prediction_length = 8 (약 2개월 앞)
- 권역 10개를 함께 학습

In [ ]:
from autogluon.timeseries import TimeSeriesPredictor

prediction_length = 8
train_data, test_data = ts.train_test_split(prediction_length)

predictor = TimeSeriesPredictor(
    prediction_length=prediction_length,
    target="target",
    freq="W-SAT",
).fit(train_data, presets="medium_quality", time_limit=600)

In [ ]:
predictor.leaderboard(test_data)

## 6. 해석

In [ ]:
predictions = predictor.predict(train_data)

predictor.plot(
    data=test_data,
    predictions=predictions,
    item_ids=["Region 1"],
    quantile_levels=[0.1, 0.9],
    max_history_length=52,
)
plt.show()

### 6-1. 생각해 볼 점

- 유행 규모(피크 높이)를 맞히는가, 시점만 맞히는가
  → 계절 패턴은 잡아도 그 해 유행 강도는 예측이 어려움
- 보고율이 낮았던 기간이 학습에 포함되면?
  → 그 기간을 빼고 학습해 비교해 볼 것
- 권역을 함께 학습(전역)한 것이 유리했는가
  → 권역별 유행 시점이 비슷하면 서로 도움

- 병상 수를 공변량으로 넣으면 도움이 될까?
  → 미래 병상 수는 알 수 있는 값인지 판단 필요

## 정리

- 천 단위 쉼표 → thousands="," 필수 (없으면 학습 불가)
- 컬럼 322개 → 명명 규칙으로 필요한 것만 선택
- 지리 층위(주·권역·전국) 섞지 말 것
- 보고율 변동 = 구조적 단절 → 값 해석에 주의
- 주간 데이터 → freq="W-SAT", 집계 불필요

- 이 데이터의 교훈: **데이터의 '생산 과정'을 알아야 한다**
  → 값이 변한 게 현실 변화인지 측정 방식 변화인지
    구분하는 것은 사람의 몫